# 04 - Tracking

Tracking sobre video y análisis simple de trayectorias. No hay objetos `Path` ni configuración implícita.


In [ ]:
import os

NOTEBOOK_DIR = os.path.abspath("notebooks")
OUTPUT_DIR = os.path.join(NOTEBOOK_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE = os.path.join(NOTEBOOK_DIR, "bus.jpg")
if not os.path.exists(IMAGE):
    IMAGE = "https://ultralytics.com/images/bus.jpg"

import pandas as pd

from vision.yolo.track import (
    track_video,
    build_tracks_dataframe,
    compute_track_statistics,
    smooth_tracks,
    filter_short_tracks,
)
from vision.yolo.plotting import (
    plot_tracking_trajectories,
    plot_video_statistics,
)
from vision.yolo.video import get_video_info

MODEL = "yolo11n.pt"
VIDEO = os.path.join(NOTEBOOK_DIR, "input", "4540398-hd_1080_1920_25fps.mp4")


In [ ]:
if os.path.exists(VIDEO):
    info = get_video_info(VIDEO)
    for key, value in info.items():
        print(f"{key}: {value}")
else:
    print("Video de ejemplo no encontrado:", VIDEO)


## Tracking real

Esta celda solo corre si el video existe.


In [ ]:
if os.path.exists(VIDEO):
    raw = track_video(
        MODEL,
        VIDEO,
        tracker="bytetrack.yaml",
        confidence=0.25,
        save_to=os.path.join(OUTPUT_DIR, "04_raw_tracks.parquet"),
    )
    display(raw.head())


## Ejemplo sintético

Permite probar el análisis aunque no tengas un video disponible.


In [ ]:
raw_example = pd.DataFrame({
    "frame": [0, 1, 2, 0, 1, 2],
    "track_id": [1, 1, 1, 2, 2, 2],
    "class_name": ["person"] * 6,
    "confidence": [0.90, 0.88, 0.86, 0.80, 0.82, 0.84],
    "xmin": [10, 20, 30, 100, 95, 90],
    "ymin": [20, 25, 30, 60, 65, 70],
    "xmax": [60, 70, 80, 150, 145, 140],
    "ymax": [120, 125, 130, 160, 165, 170],
    "timestamp": [0.00, 0.03, 0.06, 0.00, 0.03, 0.06],
})

tracks = build_tracks_dataframe(raw_example)
tracks.head()


In [ ]:
stats = compute_track_statistics(tracks)
stats


In [ ]:
long_tracks = filter_short_tracks(tracks, min_frames=3)
smoothed = smooth_tracks(long_tracks, window=2)

plot_tracking_trajectories(
    smoothed,
    image_width=200,
    image_height=200,
    save_to=os.path.join(OUTPUT_DIR, "04_tracking_trajectories.png"),
)

plot_video_statistics(
    raw_example,
    save_to=os.path.join(OUTPUT_DIR, "04_tracking_video_stats.png"),
)
